In [1]:
from tkinter import *
from tkinter import ttk
from matplotlib.figure import Figure
from matplotlib.backends.backend_tkagg import FigureCanvasTkAgg
import math

def f(x, y):
    return (x + y) * (math.cos((x + y) ** 2)) ** 2

def rk4_step(f, x, y, h):
    k1 = f(x, y)
    k2 = f(x + h / 2, y + h / 2 * k1)
    k3 = f(x + h / 2, y + h / 2 * k2)
    k4 = f(x + h, y + h * k3)
    return y + h / 6 * (k1 + 2 * k2 + 2 * k3 + k4)

def start():
    global interrupt, x_vals, y_vals, h, eps, x_end
    try:
        y0_val = float(y0_var.get())
    except ValueError:
        return

    interrupt = False
    plot_button.config(text="Stop", command=stop)

    ax.clear()
    ax.grid()

    x_vals = [0.0]
    y_vals = [y0_val]
    x_end = 2.5
    eps = 1e-10
    h = 0.1

    root.after(1, step)

def stop():
    global interrupt
    interrupt = True

def step():
    global x_vals, y_vals, h
    if interrupt:
        finish()
        return

    x_cur = x_vals[-1]
    y_cur = y_vals[-1]

    if x_cur >= x_end - 1e-12:
        finish()
        return

    if x_cur + h > x_end:
        h = x_end - x_cur

    y1 = rk4_step(f, x_cur, y_cur, h)
    y_half = rk4_step(f, x_cur, y_cur, h / 2)
    y2 = rk4_step(f, x_cur + h / 2, y_half, h / 2)

    err = abs(y1 - y2)

    if err <= eps:
        x_vals.append(x_cur + h)
        y_vals.append(y2)
        h = min(h * 2.0, x_end - x_vals[-1])
        if h < 1e-12:
            finish()
            return
        ax.clear()
        ax.plot(x_vals, y_vals)
        ax.grid()
        canvas.draw_idle()
        root.after(1, step)
    else:
        h = h / 2
        if h < 1e-14:
            finish()
            return
        root.after(1, step)

def finish():
    plot_button.config(text="Start", command=start)
    global interrupt
    interrupt = False

root = Tk()
root.title("Решение дифференциального уравнения")

mainframe = ttk.Frame(root, padding=(5, 8))
mainframe.grid(column=0, row=0, sticky=(N, W, E, S))
root.columnconfigure(0, weight=1)
root.rowconfigure(0, weight=1)
mainframe.columnconfigure(0, weight=1)
mainframe.rowconfigure(1, weight=1)

controls_frame = ttk.Frame(mainframe)
controls_frame.grid(column=0, row=0, sticky=(W, E), pady=5)

ttk.Label(controls_frame, text="y(0):").grid(column=0, row=0, padx=5)
y0_var = StringVar(value="0.0")
y0_entry = ttk.Entry(controls_frame, textvariable=y0_var, width=8)
y0_entry.grid(column=1, row=0, padx=5)

plot_button = ttk.Button(controls_frame, text="Start", command=start)
plot_button.grid(column=2, row=0, padx=10)

fig = Figure()
ax = fig.add_subplot(111)
canvas = FigureCanvasTkAgg(fig, mainframe)
canvas_widget = canvas.get_tk_widget()
canvas_widget.grid(column=0, row=1, sticky=(N, W, E, S))

y0_entry.focus()
root.bind("<Return>", lambda e: start())

interrupt = False
x_vals = []
y_vals = []
h = 0.0
eps = 0.0
x_end = 0.0

root.mainloop()